# 7B 线步骤①冒烟 v3（Kaggle，CPU 验证轮 / GPU 实测轮共用）

CPU 轮（enable_gpu=false）：验证 pip 链路 + clone + tokenizer + 权重下载，冒烟格自动跳过。
GPU 轮（metadata 翻 enable_gpu=true 后重推）：全链路跑，产出 steady tok/s。
所有关键步骤 tee 到 /kaggle/working（失败可 kaggle kernels output 拉回日志）。

In [ ]:
# Cell 1a: torch 升级（>=2.13 过 gptqmodel 安全检查；cu128 轮在 GPU 轮复用）
!pip install -q "torch>=2.13,<2.16" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tee /kaggle/working/pip_torch.log | tail -3
!python -c "import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())" 2>&1 | tee /kaggle/working/torch_check.txt

In [ ]:
# Cell 1b: 其余依赖（gptqmodel cp313 wheel 是否存在在此见分晓）
!pip install -q -r requirements.txt 2>&1 | tee /kaggle/working/pip_req.log | tail -5
!python -c "import transformers, gptqmodel; print('transformers', transformers.__version__, '| gptqmodel', gptqmodel.__version__)" 2>&1 | tee /kaggle/working/import_check.txt
!python -c "import torch; print('torch 仍未被替换:', torch.__version__, torch.cuda.is_available())" 2>&1 | tee -a /kaggle/working/import_check.txt

In [ ]:
# Cell 2: 浅克隆项目（公开仓库）
import os
os.environ['GIT_TERMINAL_PROMPT'] = '0'
!git clone --depth 1 https://github.com/Huzhenghua/emotion_model.git 2>&1 | tee /kaggle/working/clone.log
%cd emotion_model
!git log --oneline -1

In [ ]:
# Cell 3: AWQ 权重下载（~5GB，两轮都跑，证明下载链路）
!modelscope download --model Qwen/Qwen2.5-7B-Instruct-AWQ --local_dir Qwen2.5-7B-Instruct-AWQ 2>&1 | tee /kaggle/working/download.log | tail -3
!ls Qwen2.5-7B-Instruct-AWQ/

In [ ]:
# Cell 4: tokenizer + 导入链验证（CPU 安全）
!python -c "
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Qwen2.5-7B-Instruct-AWQ')
print('tokenizer OK, vocab', len(tok))
" 2>&1 | tee /kaggle/working/tokenizer_check.txt

In [ ]:
# Cell 5: 步骤①冒烟（仅 GPU 轮真正执行；CPU 轮跳过）
import torch, subprocess
if torch.cuda.is_available():
    r = subprocess.run('python eval/smoke_7b_load.py --model Qwen2.5-7B-Instruct-AWQ --max-new-tokens 128',
                       shell=True, capture_output=False)
else:
    print('[CPU 验证轮] 冒烟跳过——CUDA 不可用，符合预期。GPU 轮重推后执行。')
    with open('/kaggle/working/cpu_validation_passed.txt', 'w') as f:
        f.write('env + clone + tokenizer + download OK; smoke skipped (no GPU)')

In [ ]:
# Cell 6: 产出归集
!ls -la /kaggle/working/